In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import json
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from google.colab import drive

drive.mount('/content/drive')

MODEL_NAME    = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_Q_LENGTH  = 128
MAX_C_LENGTH  = 384
BATCH_SIZE    = 16
C       = 10.0
CLASS_WEIGHT  = 'balanced'
CTX_LIMIT     = 3
POOLING       = 'mean'
COMBINE_MODES = ['concat', 'diff', 'prod', 'concat_diff']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()

def load_and_preprocess(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rows = []
    label_mapping = {"yes": 1, "no": 0, "maybe": 2}
    for pmid, item in data.items():
        question = item["QUESTION"]
        contexts = " ".join(item["CONTEXTS"][:CTX_LIMIT])
        rows.append({
            "pmid"    : pmid,
            "question": question,
            "context" : contexts,
            "label"   : label_mapping[item["final_decision"]]
        })
    return pd.DataFrame(rows)

def encode_texts(texts, max_length):
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        encoded = tokenizer(
            batch,
            max_length     = max_length,
            truncation     = True,
            padding        = 'max_length',
            return_tensors = 'pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = bert_model(**encoded)
        if POOLING == 'cls':
            embeddings = outputs.last_hidden_state[:, 0, :]
        else:
            mask = encoded['attention_mask'].unsqueeze(-1)
            embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

def get_embeddings(df):
    q_vecs = encode_texts(df['question'].tolist(), MAX_Q_LENGTH)
    c_vecs = encode_texts(df['context'].tolist(),  MAX_C_LENGTH)
    return q_vecs, c_vecs

def combine_vectors(q_vecs, c_vecs, mode):
    if mode == 'concat':
        return np.concatenate([q_vecs, c_vecs], axis=1)
    elif mode == 'diff':
        return q_vecs - c_vecs
    elif mode == 'prod':
        return q_vecs * c_vecs
    elif mode == 'concat_diff':
        return np.concatenate([q_vecs, c_vecs, q_vecs - c_vecs], axis=1)
    elif mode == 'all':
        return np.concatenate([q_vecs, c_vecs, q_vecs - c_vecs, q_vecs * c_vecs], axis=1)

print("\nEncoding all folds...")
encode_start = time.time()

all_q_train, all_c_train = [], []
all_q_dev,   all_c_dev   = [], []
all_y_train, all_y_dev   = [], []

for fold in range(10):
    print(f"  Encoding fold {fold}...")
    fold_dir = f'/content/drive/MyDrive/data/pqal_fold{fold}'
    train_df = load_and_preprocess(f'{fold_dir}/train_set.json')
    dev_df   = load_and_preprocess(f'{fold_dir}/dev_set.json')

    q_train, c_train = get_embeddings(train_df)
    q_dev,   c_dev   = get_embeddings(dev_df)

    all_q_train.append(q_train)
    all_c_train.append(c_train)
    all_q_dev.append(q_dev)
    all_c_dev.append(c_dev)
    all_y_train.append(train_df['label'].values)
    all_y_dev.append(dev_df['label'].values)

total_encode_time = time.time() - encode_start
print(f"\nAll folds encoded in {total_encode_time:.1f} seconds")

summary = {}

for mode in COMBINE_MODES:
    all_accuracies    = []
    all_reports       = []
    all_conf_matrices = []
    train_times       = []
    inference_times   = []

    for fold in range(10):
        X_train = combine_vectors(all_q_train[fold], all_c_train[fold], mode)
        X_dev   = combine_vectors(all_q_dev[fold],   all_c_dev[fold],   mode)

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_dev   = scaler.transform(X_dev)

        t0 = time.time()
        model = LogisticRegression(
            max_iter = 1000,
            C = C,
            class_weight = CLASS_WEIGHT,
            solver = 'lbfgs'
        )
        model.fit(X_train, all_y_train[fold])
        train_times.append(time.time() - t0)

        t1 = time.time()
        preds = model.predict(X_dev)
        inference_times.append(time.time() - t1)

        all_accuracies.append(accuracy_score(all_y_dev[fold], preds))
        all_reports.append(classification_report(
            all_y_dev[fold], preds,
            target_names=['no', 'yes', 'maybe'],
            output_dict=True
        ))
        all_conf_matrices.append(confusion_matrix(all_y_dev[fold], preds))

    summary[mode] = {
        'accuracies'     : all_accuracies,
        'reports'        : all_reports,
        'conf_matrices'  : all_conf_matrices,
        'train_times'    : train_times,
        'inference_times': inference_times,
    }

print("\n" + "="*60)
print("="*60)

for mode in COMBINE_MODES:
    s = summary[mode]
    accs      = s['accuracies']
    reports   = s['reports']
    cms       = s['conf_matrices']
    t_times   = s['train_times']
    i_times   = s['inference_times']
    macro_f1s = [r['macro avg']['f1-score'] for r in reports]

    print(f"\nCombination mode: {mode}")
    print(f"10-Fold Summary:")
    print(f"Mean Accuracy : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    for label in ['no', 'yes', 'maybe']:
        f1s = [r[label]['f1-score'] for r in reports]
        print(f"Mean F1 [{label:>5s}]: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Mean Macro-F1 : {np.mean(macro_f1s):.4f} ± {np.std(macro_f1s):.4f}")
    print(f"Train time    : {np.mean(t_times):.2f}s / fold  (total {np.sum(t_times):.1f}s)")
    print(f"Inference time: {np.mean(i_times)*1000:.1f}ms / fold")

    avg_cm = np.sum(cms, axis=0)
    print(f"Confusion Matrix (10-fold cumulative):")
    print(f"             Pred no  Pred yes  Pred maybe")
    for i, row_label in enumerate(['no', 'yes', 'maybe']):
        print(f"Actual {row_label:>5s}:  {avg_cm[i][0]:5d}   {avg_cm[i][1]:5d}   {avg_cm[i][2]:7d}")

best_mode = max(summary, key=lambda m: np.mean(
    [r['macro avg']['f1-score'] for r in summary[m]['reports']]
))
best_macro = np.mean([r['macro avg']['f1-score'] for r in summary[best_mode]['reports']])
best_acc   = np.mean(summary[best_mode]['accuracies'])

print(f"\n{'='*60}")
print(f"Best combination mode : {best_mode}")
print(f"  Accuracy            : {best_acc:.4f}")
print(f"  Macro F1            : {best_macro:.4f}")
print(f"  Total encoding time : {total_encode_time:.1f}s")
print(f"{'='*60}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Encoding all folds...
  Encoding fold 0...
  Encoding fold 1...
  Encoding fold 2...
  Encoding fold 3...
  Encoding fold 4...
  Encoding fold 5...
  Encoding fold 6...
  Encoding fold 7...
  Encoding fold 8...
  Encoding fold 9...

All folds encoded in 131.0 seconds


Combination mode: concat
10-Fold Summary:
Mean Accuracy : 0.5280 ± 0.0515
Mean F1 [   no]: 0.4613 ± 0.0643
Mean F1 [  yes]: 0.6418 ± 0.0569
Mean F1 [maybe]: 0.1554 ± 0.1343
Mean Macro-F1 : 0.4195 ± 0.0594
Train time    : 0.23s / fold  (total 2.3s)
Inference time: 0.5ms / fold
Confusion Matrix (10-fold cumulative):
             Pred no  Pred yes  Pred maybe
Actual    no:     78      69        22
Actual   yes:     72     177        27
Actual maybe:     18      28         9

Combination mode: diff
10-Fold Summary:
Mean Accuracy : 0.5320 ± 0.0546
Mean F1 [   no]: 0.4624 ± 0.0860
Mean F1 [  yes]: 0.6369 ± 0.0547
Mean F1 [maybe]: 0.2143 ± 0.1625
Mean Macro-F1 : 0.4379 ± 0.0718
Train time    : 0.12s / fold  (total 1.2s)
Infere